### <b>Asset definition</b>

In [ ]:
from importlib import reload
from helpers import *e

In [8]:


ticker_universe = {
    # US equities — broad & factor
    "SPY":  "S&P 500",
    "QQQ":  "Nasdaq 100 (growth/tech)",
    "IWM":  "Russell 2000 (small-cap blend)",
    "AVUV": "Avantis US Small-Cap Value (small-value factor)",
    "MTUM": "iShares MSCI USA Momentum",
    "USMV": "iShares MSCI USA Min Vol",
    
    "VT":  "Global All-Cap (total world market)",

    # International equities
    "VEA":  "Developed Markets ex-US",
    "VWO":  "Emerging Markets",

    # Bonds — duration spectrum
    "VGSH": "Short-Term Treasury (1-3y)",
    "IEF":  "Intermediate Treasury (7-10y)",
    "VGLT": "Long-Term Treasury (20-30y)",
    "TIP":  "TIPS (inflation-linked)",
    "AGG":  "US Aggregate Bond",
    "HYG":  "High Yield Corporate",

    # Commodities & real assets
    "GLD":  "Gold",
    "SLV":  "Silver",
    "GSG":  "Broad Commodities (S&P GSCI)",
    "VNQ":  "US REITs",
}

interest_tickers = [
    "SPY", "QQQ", "VT", "VEA", "VWO", "VGSH", "IEF", "VGLT", "AGG", "GLD"
]

#download data for the tickers in the universe
start_date, end_date = common_window(interest_tickers)
today = pd.Timestamp.today().strftime("%Y-%m-%d")

print(start_date, end_date)

LOOKBACK_YEARS = 2
REBALANCE_FREQ = "MS"

for ticker in interest_tickers:
    load_data(ticker, start_date, end_date)

FileNotFoundError: [Errno 2] No such file or directory: '/Users/giacomomaggiore/Desktop/coding/data/SPY.csv'

In [ ]:
# Covariance configuration
COV_METHOD = "shrunk"   # 'shrunk' | 'empirical' | 'oas' | 'ewma' | 'factor'
COV_PARAMS = {}         # e.g., {'span': 60} for ewma, {'n_factors': 3} for factor

In [ ]:
def build_portfolios(tickers, start, end, lookback_years=LOOKBACK_YEARS, freq=REBALANCE_FREQ, cov_method=None, cov_params=None):
    # Use business-month start to avoid holidays
    # offset start by lookback_years to ensure we have enough data for the first rebalance
    start = pd.Timestamp(start) + pd.DateOffset(years=lookback_years)
    rebalance_dates = pd.date_range(start, end, freq="BMS")

    # default cov settings from globals if not provided
    if cov_method is None:
        cov_method = COV_METHOD
    if cov_params is None:
        cov_params = COV_PARAMS

    mv, ms, mc = {}, {}, {}
    for dt in rebalance_dates:
        as_of = dt.date()
        mv[dt] = min_variance(tickers, as_of=as_of, timeframe_years=lookback_years, cov_method=cov_method, cov_params=cov_params)
        ms[dt] = max_sharpe(tickers, as_of=as_of, timeframe_years=lookback_years, cov_method=cov_method, cov_params=cov_params)
        mc[dt] = pd.Series({t: 1.0 if t == "VT" else 0.0 for t in tickers})

    def to_daily(d):
        df = pd.DataFrame(d).T.sort_index()
        # Start the daily index at the first available weight to avoid leading NaNs
        first = df.index.min()
        daily_idx = pd.bdate_range(first, end)
        return df.reindex(daily_idx, method="ffill")

    return {
        "min_variance": to_daily(mv),
        "max_sharpe":   to_daily(ms),
        "market_cap":   to_daily(mc),
    }

In [ ]:
portfolios = build_portfolios(interest_tickers, start_date, end_date, cov_method=COV_METHOD, cov_params=COV_PARAMS)

portfolios
# portfolios["min_variance"], portfolios["max_sharpe"], portfolios["market_cap"]
# each is a DataFrame: index=business days, columns=tickers, values=weights


TypeError: min_variance() got an unexpected keyword argument 'cov_method'

In [ ]:
portfolios["min_variance"]

,SPY,QQQ,VT,VEA,VWO,VGSH,IEF,VGLT,AGG,GLD
2012-02-01,0.1,0.1,0.1,0.1,0.1,0.1,0.1,0.1,0.1,0.1
2012-02-02,0.1,0.1,0.1,0.1,0.1,0.1,0.1,0.1,0.1,0.1
2012-02-03,0.1,0.1,0.1,0.1,0.1,0.1,0.1,0.1,0.1,0.1
2012-02-06,0.1,0.1,0.1,0.1,0.1,0.1,0.1,0.1,0.1,0.1
2012-02-07,0.1,0.1,0.1,0.1,0.1,0.1,0.1,0.1,0.1,0.1
...,...,...,...,...,...,...,...,...,...,...
2026-06-26,0.1,0.1,0.1,0.1,0.1,0.1,0.1,0.1,0.1,0.1
2026-06-29,0.1,0.1,0.1,0.1,0.1,0.1,0.1,0.1,0.1,0.1
2026-06-30,0.1,0.1,0.1,0.1,0.1,0.1,0.1,0.1,0.1,0.1
2026-07-01,0.1,0.1,0.1,0.1,0.1,0.1,0.1,0.1,0.1,0.1
